In [ ]:
import re
from collections import Counter
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [ ]:
import re

file_path = "rus.txt"
sentence_pairs = []

with open(file_path, encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) > 1:
            sentence_pairs.append((parts[0], parts[1]))

print(sentence_pairs[:20])
print("Total pairs:", len(sentence_pairs))


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-zа-яё\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


clean_pairs = [(normalize(en), normalize(ru)) for en, ru in sentence_pairs]

print(clean_pairs[:20])

[('Go.', 'Марш!'), ('Go.', 'Иди.'), ('Go.', 'Идите.'), ('Hi.', 'Здравствуйте.'), ('Hi.', 'Привет!'), ('Hi.', 'Хай.'), ('Hi.', 'Здрасте.'), ('Hi.', 'Здоро́во!'), ('Run!', 'Беги!'), ('Run!', 'Бегите!'), ('Run.', 'Беги!'), ('Run.', 'Бегите!'), ('Who?', 'Кто?'), ('Wow!', 'Вот это да!'), ('Wow!', 'Круто!'), ('Wow!', 'Здорово!'), ('Wow!', 'Ух ты!'), ('Wow!', 'Ого!'), ('Wow!', 'Вах!'), ('Fire!', 'Огонь!')]
363386


In [ ]:
processed_pairs = [
    (en, f"<sos> {ru} <eos>")
    for en, ru in clean_pairs
]

print(processed_pairs[:20])

[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>'), ('hi', '<sos> хай <eos>'), ('hi', '<sos> здрасте <eos>'), ('hi', '<sos> здорово <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('who', '<sos> кто <eos>'), ('wow', '<sos> вот это да <eos>'), ('wow', '<sos> круто <eos>'), ('wow', '<sos> здорово <eos>'), ('wow', '<sos> ух ты <eos>'), ('wow', '<sos> ого <eos>'), ('wow', '<sos> вах <eos>'), ('fire', '<sos> огонь <eos>')]


In [ ]:
from collections import Counter

def create_vocab(sent_list):
    counter = Counter()

    for sent in sent_list:
        counter.update(sent.split())

    vocab = {"<pad>": 0, "<oov>": 1}

    for word in counter.keys():
        vocab[word] = len(vocab)

    return vocab


eng_sentences = [en for en, _ in processed_pairs]
rus_sentences = [ru for _, ru in processed_pairs]

eng_vocab = create_vocab(eng_sentences)
rus_vocab = create_vocab(rus_sentences)

print("Eng vocab size:", len(eng_vocab))
print("Rus vocab size:", len(rus_vocab))

In [ ]:
def encode_sentence(sentence, vocab):
    tokens = sentence.split() if isinstance(sentence, str) else sentence
    return [vocab.get(tok, vocab["<oov>"]) for tok in tokens]


indexed_data = [
    (encode_sentence(en, eng_vocab),
     encode_sentence(ru, rus_vocab))
    for en, ru in processed_pairs
]

print(indexed_data[:20])

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    indexed_data,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(train_data))
print("Val size:", len(val_data))

[([2], [2, 3, 4]), ([2], [2, 5, 4]), ([2], [2, 6, 4]), ([3], [2, 7, 4]), ([3], [2, 8, 4]), ([3], [2, 9, 4]), ([3], [2, 10, 4]), ([3], [2, 11, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([5], [2, 14, 4]), ([6], [2, 15, 16, 17, 4]), ([6], [2, 18, 4]), ([6], [2, 11, 4]), ([6], [2, 19, 20, 4]), ([6], [2, 21, 4]), ([6], [2, 22, 4]), ([7], [2, 23, 4])]


### Data Loaders + dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class SeqDataset(Dataset):
    def __init__(self, pairs):
        self.data = pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, trg = self.data[idx]
        return torch.tensor(src), torch.tensor(trg)


def collate(batch):
    src_batch = [item[0] for item in batch]
    trg_batch = [item[1] for item in batch]

    src_pad = pad_sequence(src_batch, padding_value=0)
    trg_pad = pad_sequence(trg_batch, padding_value=0)

    return src_pad, trg_pad


batch_size = 32

train_loader = DataLoader(
    SeqDataset(train_data),
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    SeqDataset(val_data),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate
)

for s, t in train_loader:
    print(s.shape, t.shape)
    break

### Model encoder-decoder

In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embed = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, hid_dim)

    def forward(self, src):
        emb = self.embed(src)
        _, (hidden, cell) = self.rnn(emb)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embed = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, output_dim)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(0)
        emb = self.embed(x)
        out, (hidden, cell) = self.rnn(emb, (hidden, cell))
        pred = self.fc(out.squeeze(0))
        return pred, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, device):
        super().__init__()
        self.encoder = enc
        self.decoder = dec
        self.device = device

    def forward(self, src, trg, teacher_ratio=0.5):
        T, B = trg.shape
        vocab_size = self.decoder.output_dim

        outputs = torch.zeros(T, B, vocab_size).to(self.device)

        hidden, cell = self.encoder(src)
        inp = trg[0]

        for t in range(1, T):
            out, hidden, cell = self.decoder(inp, hidden, cell)
            outputs[t] = out

            best = out.argmax(1)
            use_teacher = torch.rand(1).item() < teacher_ratio
            inp = trg[t] if use_teacher else best

        return outputs

In [ ]:
input_dim = len(eng_vocab)
output_dim = len(rus_vocab)

model = Seq2Seq(
    Encoder(input_dim, 256, 1024),
    Decoder(output_dim, 256, 1024),
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

device = model.device
model.to(device)

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=rus_vocab["<pad>"])

### Training

In [ ]:
def train_epoch(model, loader):
    model.train()
    total = 0

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()
        out = model(src, trg)

        out_dim = out.shape[-1]
        out = out[1:].view(-1, out_dim)
        trg = trg[1:].reshape(-1)

        loss = criterion(out, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total += loss.item()

    return total / len(loader)


def eval_epoch(model, loader):
    model.eval()
    total = 0

    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)

            out = model(src, trg, teacher_ratio=0)

            out_dim = out.shape[-1]
            out = out[1:].view(-1, out_dim)
            trg = trg[1:].reshape(-1)

            total += criterion(out, trg).item()

    return total / len(loader)

In [ ]:
import time

best_loss = float("inf")

for epoch in range(10):
    start = time.time()

    train_loss = train_epoch(model, train_loader)
    val_loss = eval_epoch(model, val_loader)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), f"best_model_{epoch}.pt")

    print(f"\nEpoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)

Epoch 1
Train loss: 3.502577837811241
Val loss: 2.887019557694734
Epoch 2
Train loss: 2.0173331161285404
Val loss: 2.5914793418536726
Epoch 3
Train loss: 1.546307832071528
Val loss: 2.5460400739291185
Epoch 4
Train loss: 1.2811346857837747
Val loss: 2.5563819178603064
Epoch 5
Train loss: 1.1271598233845972
Val loss: 2.6026149433053716
Epoch 6
Train loss: 1.0297106555450337
Val loss: 2.630908243777886
Epoch 7
Train loss: 0.9596692218712286
Val loss: 2.689044909997725
Epoch 8
Train loss: 0.9027941356698562
Val loss: 2.725216087163754
Epoch 9
Train loss: 0.8624668633996883
Val loss: 2.7473855209077747
Epoch 10
Train loss: 0.8281964540547175
Val loss: 2.80197535591646


In [ ]:
import torch.nn.functional as F

def beam_translate(sentence, beam_size=5, max_len=50):
    model.eval()

    tokens = sentence.lower().split()
    src_idx = [eng_vocab.get(t, eng_vocab["<oov>"]) for t in tokens]

    src_tensor = torch.LongTensor(src_idx).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    beams = [([rus_vocab["<sos>"]], 0.0, hidden, cell)]

    for _ in range(max_len):
        candidates = []

        for seq, score, h, c in beams:
            if seq[-1] == rus_vocab["<eos>"]:
                candidates.append((seq, score, h, c))
                continue

            inp = torch.LongTensor([seq[-1]]).to(device)

            with torch.no_grad():
                out, h_new, c_new = model.decoder(inp, h, c)

            probs = F.log_softmax(out, dim=1)
            top_p, top_i = probs.topk(beam_size)

            for i in range(beam_size):
                token = top_i[0][i].item()
                new_score = score + top_p[0][i].item()
                candidates.append((seq + [token], new_score, h_new, c_new))

        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_size]

    inv_vocab = {v: k for k, v in rus_vocab.items()}
    best_seq = beams[0][0]

    return [inv_vocab[i] for i in best_seq][1:]

In [ ]:
test_sentences = [
    "hello how are you",
    "i love programming",
    "where is the cinema",
    "this is a good idea",
    "please show me the path",
    "can you help me"
]

for s in test_sentences:
    result = beam_translate(s)
    print("EN:", s)
    print("RU:", " ".join(result))
    print()

English: hello how are you
Russian: как ты <eos>

English: i love programming
Russian: я люблю тома <eos>

English: where is the cinema
Russian: где находится <eos>

English: this is a good idea
Russian: хорошая идея <eos>

English: please show me the path
Russian: пожалуйста покажите мне дорогу <eos>

English: can you help me
Russian: вы можете мне помочь <eos>

